# Waaiya screen blur: train the woman/man person classifier v2

Person crops from the **Open Images V7 train split** (Woman / Man / Girl / Boy boxes, IsGroupOf = 0; boxes
CC BY 4.0, images listed as CC BY 2.0), cut from images resized to the phone's 640-px screen frame. Two small
backbones (each at most ~4x the cost of v1's MobileNetV3-small at 192x128) are fine-tuned from ImageNet weights
(Hugging Face safetensors). The MIAP extension is **not** used (its authors ask that no gender classifier be built
from it). Checkpoints are chosen on a 1% holdout of train images; the validation split stays untouched for the
end-to-end scorer.

Outputs in `/kaggle/working/out`: `gender_v2_<arch>.pt` (state_dict + labels + size + mean/std + arch),
`gender_v2_<arch>_eval.json`, logs.

In [ ]:
PREP_MINUTES = 40     # stop fetching images after this (train on what arrived)
TRAIN_MINUTES = 100   # per model when two GPUs train side by side; split in two on one GPU
PER_SIDE = 200000       # crops wanted per side (woman+girl, man+boy)
ARCHS = [('mobilenetv3_large_100', 'miil_in21k_ft_in1k'), ('mobilenetv4_conv_small', 'e2400_r224_in1k')]
CSV_URL = 'https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv'
CSV_ROWS = 2470641

import os, subprocess, time, glob, shutil
CODE, TMP, OUT = "/kaggle/working/code", "/kaggle/tmp", "/kaggle/working/out"
for d in (CODE + "/models/vision", TMP, OUT):
    os.makedirs(d, exist_ok=True)

def run(cmd, cwd=CODE, env=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(["bash", "-o", "pipefail", "-c", cmd], cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
%%writefile /kaggle/working/code/models/vision/prep_train_crops.py
"""Person crops from the Open Images V7 *train* split for the woman/man classifier v2 (boxes CC BY 4.0).

Crops are cut the way the phone sees people: each image is first resized to long side 640 (the screen frame the
blur works on), then every Woman / Man / Girl / Boy box (IsGroupOf = 0) is cut with 25% extra margin on each side
(train_gender_v2.py jitters the box and applies the runtime's 10% context inside that margin). Crops keep the
frame's pixels (no upscaling; long side capped at 256).

    python models/vision/prep_train_crops.py --csv train-wmgb.csv --out /kaggle/tmp/gcrops \
        [--per-side 200000] [--max-minutes 50] [--limit 0]

--csv: the train box CSV, or only its Woman/Man/Girl/Boy rows with the header line.
Images are split by md5(image id): 1 in 100 goes to the *holdout* (checkpoint and threshold selection, never
trained on). Training images are taken in a fixed random order until both sides (woman+girl, man+boy) reach
--per-side crops. Holdout images are fetched first. Writes <out>/index.csv:
    image, label, holdout, bx0, by0, bx1, by1 (box inside the crop, 0..1), h_frame (box height / frame), crop
"""
import argparse
import csv
import hashlib
import io
import os
import random
import sys
import time
import urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool

from PIL import Image

LABELS = {"/m/03bt1vf": "woman", "/m/04yx4": "man", "/m/05r655": "girl", "/m/01bl7v": "boy"}
FEMALE = {"woman", "girl"}
URL = "https://open-images-dataset.s3.amazonaws.com/{split}/{id}.jpg"
FRAME_LONG = 640  # the blur's screen frame (long side)
MARGIN = 0.25     # saved context around the box, each side, fraction of the box size
CROP_LONG = 256
MIN_W, MIN_H = 6, 10  # skip people smaller than this in the 640 frame (pixels)


def is_holdout(image_id):
    return int(hashlib.md5(image_id.encode()).hexdigest(), 16) % 100 == 0


def read_boxes(path):
    by_image = defaultdict(list)
    n = 0
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            lab = LABELS.get(r["LabelName"])
            if lab is None or r["IsGroupOf"] != "0":
                continue
            n += 1
            by_image[r["ImageID"]].append((lab, float(r["XMin"]), float(r["YMin"]), float(r["XMax"]), float(r["YMax"])))
    return by_image, n


def select(by_image, per_side, seed=0):
    ids = sorted(by_image)
    hold = [i for i in ids if is_holdout(i)]
    rest = [i for i in ids if not is_holdout(i)]
    random.Random(seed).shuffle(rest)
    nf = nm = 0
    train = []
    for i in rest:
        f = sum(b[0] in FEMALE for b in by_image[i])
        m = len(by_image[i]) - f
        if (f and nf < per_side) or (m and nm < per_side):
            train.append(i)
            nf += f
            nm += m
        if nf >= per_side and nm >= per_side:
            break
    return hold, train, nf, nm


def fetch(split, image_id, tries=3):
    for k in range(tries):
        try:
            with urllib.request.urlopen(URL.format(split=split, id=image_id), timeout=30) as r:
                return r.read()
        except Exception as e:  # noqa: BLE001 - network: retry, then give up on this image
            if k == tries - 1:
                print(f"skip {image_id}: {e}", file=sys.stderr)
                return None
            time.sleep(1 + 2 * k)


_G = {}


def _init(out, split, threads):
    _G.update(out=out, split=split, pool=ThreadPoolExecutor(threads))


def one(job):
    image_id, boxes, hold = job
    data = fetch(_G["split"], image_id)
    if data is None:
        return []
    try:
        im = Image.open(io.BytesIO(data))
        s = FRAME_LONG / max(im.size)
        W, H = max(1, round(im.size[0] * s)), max(1, round(im.size[1] * s))
        im.draft("RGB", (W, H))  # JPEG DCT downscale: faster decode, still >= the frame size
        im = im.convert("RGB").resize((W, H), Image.BILINEAR)
    except Exception as e:  # noqa: BLE001 - a broken image: skip it
        print(f"skip {image_id}: {e}", file=sys.stderr)
        return []
    rows = []
    sub = os.path.join(_G["out"], "crops", image_id[:2])
    os.makedirs(sub, exist_ok=True)
    for n, (lab, x0, y0, x1, y1) in enumerate(boxes):
        bx0, by0, bx1, by1 = x0 * W, y0 * H, x1 * W, y1 * H
        bw, bh = bx1 - bx0, by1 - by0
        if bw < MIN_W or bh < MIN_H:
            continue
        cx0, cy0 = max(0, int(bx0 - MARGIN * bw)), max(0, int(by0 - MARGIN * bh))
        cx1, cy1 = min(W, int(round(bx1 + MARGIN * bw))), min(H, int(round(by1 + MARGIN * bh)))
        c = im.crop((cx0, cy0, cx1, cy1))
        cw, ch = c.size
        k = CROP_LONG / max(cw, ch)
        if k < 1:
            c = c.resize((max(1, round(cw * k)), max(1, round(ch * k))), Image.BILINEAR)
        name = f"crops/{image_id[:2]}/{image_id}_{n}.jpg"
        c.save(os.path.join(_G["out"], name), quality=90)
        rows.append([image_id, lab, int(hold), f"{(bx0 - cx0) / cw:.4f}", f"{(by0 - cy0) / ch:.4f}",
                     f"{(bx1 - cx0) / cw:.4f}", f"{(by1 - cy0) / ch:.4f}", f"{bh / H:.4f}", name])
    return rows


def batch(jobs):
    return [r for rows in _G["pool"].map(one, jobs) for r in rows]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--split", default="train")
    ap.add_argument("--per-side", type=int, default=200000)
    ap.add_argument("--limit", type=int, default=0, help="max images in total (0 = no limit; for tests)")
    ap.add_argument("--max-minutes", type=float, default=0, help="stop fetching after this time (0 = no limit)")
    ap.add_argument("--procs", type=int, default=0, help="worker processes (default 2 x CPUs)")
    ap.add_argument("--threads", type=int, default=12, help="download threads per process")
    a = ap.parse_args()

    t0 = time.time()
    by_image, n = read_boxes(a.csv)
    hold, train, nf, nm = select(by_image, a.per_side)
    print(f"{n} boxes in {len(by_image)} images; holdout {len(hold)} images; train {len(train)} images "
          f"({nf} female, {nm} male boxes); parse {time.time() - t0:.0f} s", flush=True)
    jobs = [(i, by_image[i], True) for i in hold] + [(i, by_image[i], False) for i in train]
    if a.limit:
        jobs = [j for j in jobs if j[2]][: max(1, a.limit // 10)] + [j for j in jobs if not j[2]][: a.limit - max(1, a.limit // 10)]
    os.makedirs(os.path.join(a.out, "crops"), exist_ok=True)
    chunks = [jobs[k:k + 48] for k in range(0, len(jobs), 48)]
    t0, done, counts = time.time(), 0, defaultdict(int)
    with open(os.path.join(a.out, "index.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image", "label", "holdout", "bx0", "by0", "bx1", "by1", "h_frame", "crop"])
        pool = Pool(a.procs or 2 * os.cpu_count(), _init, (a.out, a.split, a.threads))
        try:
            for rows in pool.imap(batch, chunks):  # in order: the holdout comes first and is complete
                w.writerows(rows)
                for r in rows:
                    counts[(r[1], r[2])] += 1
                done += 48
                if done % 4800 < 48:
                    f.flush()
                    print(f"{min(done, len(jobs))}/{len(jobs)} images, {sum(counts.values())} crops, "
                          f"{time.time() - t0:.0f} s", flush=True)
                if a.max_minutes and time.time() - t0 > a.max_minutes * 60:
                    print("time limit: stopping the fetch", flush=True)
                    break
        finally:
            pool.terminate()
    print("crops (label, holdout):", dict(counts), f"{time.time() - t0:.0f} s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/models/vision/train_gender_v2.py
"""Woman/man person-crop classifier v2 for the screen blur (crops from prep_train_crops.py), GPU or CPU.

Differences from train_gender.py (v1):
  - data: Open Images *train* split, crops cut from the 640-px frame; the runtime's 10% context is applied here
    around a jittered box (detector boxes are not the annotators' boxes);
  - strong low-resolution augmentation (a 16:9 video in a portrait phone shows people 3-4x smaller);
  - loss on the decision the blur makes: binary cross-entropy on p(man)+p(boy), plus a small 4-class term
    (the 4 labels stay: the C side sums man+boy); no label smoothing (it pulls confident scores down, toward the male_clear threshold);
  - checkpoint selection on a holdout of *train* images (never on the validation images the scorer uses):
    men left clear when at most 0.5% of women/girls are left clear;
  - pretrained weights only from Hugging Face safetensors (no pickles); mean/std from the timm pretrained_cfg;
  - wall-clock budget with warmup + cosine by time, EMA weights, AMP on CUDA.

    python models/vision/train_gender_v2.py --data /kaggle/tmp/gcrops --out out --arch mobilenetv3_large_100 \
        --tag miil_in21k_ft_in1k --minutes 90

Writes <out>/<name>.pt ({state_dict, labels, size, mean, std, arch}: loadable with weights_only=True, input of
export_vision.py gender) and <out>/<name>_eval.json.
"""
import argparse
import copy
import csv
import json
import math
import os
import random
import time

import numpy as np
import timm
import torch
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

LABELS = ["woman", "man", "girl", "boy"]
MALE, FEMALE = [1, 3], [0, 2]
CONTEXT = 0.10  # im_vision.c classify()


def read_index(root):
    tr, ho = [], []
    with open(os.path.join(root, "index.csv"), newline="") as f:
        for r in csv.DictReader(f):
            item = (os.path.join(root, r["crop"]), LABELS.index(r["label"]),
                    tuple(float(r[k]) for k in ("bx0", "by0", "bx1", "by1")), float(r["h_frame"]))
            (ho if r["holdout"] == "1" else tr).append(item)
    return tr, ho


class Crops(Dataset):
    """uint8 CHW crops, stretched to the model input like the C side. mode: train | eval | lowres (eval at 1/3)."""

    def __init__(self, items, size, mode):
        self.items, self.size, self.mode = items, size, mode

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, y, (bx0, by0, bx1, by1), _ = self.items[i]
        im = Image.open(path).convert("RGB")
        W, H = im.size
        x0, y0, x1, y1 = bx0 * W, by0 * H, bx1 * W, by1 * H
        bw, bh = x1 - x0, y1 - y0
        train = self.mode == "train"
        if train:  # detector-like box error: each side moves by ~5% of the box size
            x0 += random.gauss(0, 0.05) * bw
            x1 += random.gauss(0, 0.05) * bw
            y0 += random.gauss(0, 0.05) * bh
            y1 += random.gauss(0, 0.05) * bh
            if x1 - x0 < 2 or y1 - y0 < 2:
                x0, y0, x1, y1 = bx0 * W, by0 * H, bx1 * W, by1 * H
            bw, bh = x1 - x0, y1 - y0
        c = (max(0, x0 - CONTEXT * bw), max(0, y0 - CONTEXT * bh), min(W, x1 + CONTEXT * bw), min(H, y1 + CONTEXT * bh))
        im = im.crop(tuple(int(round(v)) for v in c))
        f = 1.0
        if train and random.random() < 0.5:  # smaller on screen: portrait video strips, small capture sizes
            f = math.exp(random.uniform(math.log(0.2), 0))
        elif self.mode == "lowres":
            f = 1 / 3
        if f < 1:
            im = im.resize((max(3, round(im.size[0] * f)), max(3, round(im.size[1] * f))), Image.BILINEAR)
        if train:
            if random.random() < 0.5:
                im = im.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.1:
                im = im.filter(ImageFilter.GaussianBlur(random.uniform(0.5, 1.5)))
            if random.random() < 0.05:
                im = im.convert("L").convert("RGB")
        h, w = self.size
        im = im.resize((w, h), Image.BILINEAR)  # stretched, like im_vision.c classify()
        return torch.from_numpy(np.asarray(im, np.uint8).copy()).permute(2, 0, 1), y


def to_input(x, mean, std, train):
    x = x.float() / 255.0
    if train:  # per-sample brightness / contrast / colour cast
        b = x.shape[0]
        g = torch.empty(b, 1, 1, 1, device=x.device).uniform_(0.7, 1.3)
        m = x.mean((1, 2, 3), keepdim=True)
        x = (x - m) * torch.empty(b, 1, 1, 1, device=x.device).uniform_(0.7, 1.3) + m
        x = x * g * torch.empty(b, 3, 1, 1, device=x.device).uniform_(0.92, 1.08)
        x = x.clamp(0, 1)
    return (x - mean) / std


def male_logit(logits):
    return torch.logsumexp(logits[:, MALE], 1) - torch.logsumexp(logits[:, FEMALE], 1)


def metrics(pm, y):
    female = np.isin(y, FEMALE)
    pf, pmm = pm[female], pm[~female]
    order = np.concatenate([pf, pmm]).argsort().argsort() + 1
    auc = (order[len(pf):].sum() - len(pmm) * (len(pmm) + 1) / 2) / max(1, len(pf) * len(pmm))
    rep = {"n_female": int(len(pf)), "n_male": int(len(pmm)), "auc": float(auc),
           "male_vs_female_acc": float(((pm >= 0.5) == ~female).mean()), "at_women_clear": {}, "thresholds": {}}
    for q in (0.0025, 0.005, 0.01):
        t = float(np.sort(pf)[int(math.ceil((1 - q) * len(pf))) - 1]) if len(pf) else 1.0
        rep["at_women_clear"][str(q)] = {"male_clear": t, "men_left_clear": float((pmm > t).mean())}
    for t in (0.9, 0.95, 0.98, 0.99, 0.995, 0.999):
        rep["thresholds"][str(t)] = {"women_left_clear": float((pf >= t).mean()), "men_left_clear": float((pmm >= t).mean())}
    return rep


@torch.no_grad()
def predict(net, loader, mean, std, dev, amp):
    net.eval()
    ps, ys = [], []
    for x, y in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            z = male_logit(net(to_input(x.to(dev, non_blocking=True), mean, std, False).float()).float())
        ps.append(torch.sigmoid(z).cpu())
        ys.append(y)
    return torch.cat(ps).numpy(), torch.cat(ys).numpy()


def build(arch, tag):
    """timm model with ImageNet weights from the Hugging Face safetensors file (no pickle)."""
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file

    net = timm.create_model(f"{arch}.{tag}" if tag else arch, pretrained=False, num_classes=len(LABELS))
    pc = net.pretrained_cfg
    if tag:
        sd = load_file(hf_hub_download(pc["hf_hub_id"], "model.safetensors"))
        head = pc.get("classifier", "classifier")
        sd = {k: v for k, v in sd.items() if not k.startswith(head + ".")}
        missing, unexpected = net.load_state_dict(sd, strict=False)
        assert not unexpected and all(k.startswith(head + ".") for k in missing), (missing, unexpected)
    return net, list(pc.get("mean", (0.485, 0.456, 0.406))), list(pc.get("std", (0.229, 0.224, 0.225)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--arch", default="mobilenetv3_large_100")
    ap.add_argument("--tag", default="miil_in21k_ft_in1k", help="timm pretrained tag ('' = from scratch, tests)")
    ap.add_argument("--name", default=None, help="output file stem (default gender_v2_<arch>)")
    ap.add_argument("--height", type=int, default=192)
    ap.add_argument("--width", type=int, default=128)
    ap.add_argument("--minutes", type=float, default=90, help="training wall-clock budget")
    ap.add_argument("--max-steps", type=int, default=0, help="stop after this many steps (tests)")
    ap.add_argument("--epoch-size", type=int, default=0, help="samples per epoch (default: all training crops)")
    ap.add_argument("--batch", type=int, default=256)
    ap.add_argument("--lr", type=float, default=1.5e-3)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--aux", type=float, default=0.2, help="weight of the 4-class loss")
    ap.add_argument("--ema", type=float, default=0.9995)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    random.seed(0)
    torch.manual_seed(0)
    os.makedirs(a.out, exist_ok=True)
    name = a.name or f"gender_v2_{a.arch}"
    dev = torch.device(a.device)
    amp = dev.type == "cuda"
    size = (a.height, a.width)

    train, hold = read_index(a.data)
    counts = np.bincount([y for _, y, _, _ in train], minlength=4)
    print(f"train {dict(zip(LABELS, counts.tolist()))}, holdout {len(hold)}", flush=True)
    side = np.array([counts[0] + counts[2], counts[1] + counts[3]], np.float64)
    wts = [1.0 / side[0 if y in FEMALE else 1] for _, y, _, _ in train]
    sampler = WeightedRandomSampler(wts, num_samples=a.epoch_size or len(train), replacement=True)
    kw = dict(num_workers=a.workers, pin_memory=amp, persistent_workers=a.workers > 0)
    tl = DataLoader(Crops(train, size, "train"), batch_size=a.batch, sampler=sampler, drop_last=True, **kw)
    hl = DataLoader(Crops(hold, size, "eval"), batch_size=512, **kw)
    ll = DataLoader(Crops(hold, size, "lowres"), batch_size=512, **kw)

    net, mean_l, std_l = build(a.arch, a.tag)
    print(f"{a.arch}.{a.tag}: mean {mean_l} std {std_l}", flush=True)
    net = net.to(dev).to(memory_format=torch.channels_last)
    ema = copy.deepcopy(net).eval()
    mean = torch.tensor(mean_l, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(std_l, device=dev).view(1, 3, 1, 1)
    decay, no_decay = [], []
    for n_, p in net.named_parameters():
        (no_decay if p.ndim <= 1 else decay).append(p)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": a.wd}, {"params": no_decay, "weight_decay": 0}], lr=a.lr)
    scaler = torch.cuda.amp.GradScaler(enabled=amp)
    budget = a.minutes * 60

    def lr_at(frac):  # warmup 3% of the budget, then cosine by wall time
        return frac / 0.03 if frac < 0.03 else 0.5 * (1 + math.cos(math.pi * min(1.0, (frac - 0.03) / 0.97)))

    t0, step, best, ep, done = time.time(), 0, None, 0, False
    hist = []
    while not done:
        net.train()
        tot, n = 0.0, 0
        for x, y in tl:
            frac = (time.time() - t0) / budget
            for g in opt.param_groups:
                g["lr"] = a.lr * max(0.0, lr_at(frac)) + 1e-6
            x = to_input(x.to(dev, non_blocking=True), mean, std, True).contiguous(memory_format=torch.channels_last)
            y = y.to(dev, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = net(x).float()
            loss = F.binary_cross_entropy_with_logits(male_logit(logits), (y % 2 == 1).float()) \
                + a.aux * F.cross_entropy(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            step += 1
            d = min(a.ema, (1 + step) / (10 + step))
            with torch.no_grad():
                for pe, p in zip(ema.parameters(), net.parameters()):
                    pe.mul_(d).add_(p.detach(), alpha=1 - d)
                for be, b in zip(ema.buffers(), net.buffers()):
                    be.copy_(b)
            tot += loss.item() * len(y)
            n += len(y)
            if step % 200 == 0:
                print(f"ep {ep} step {step} loss {tot / n:.4f} lr {opt.param_groups[0]['lr']:.2e} "
                      f"{time.time() - t0:.0f} s", flush=True)
            if time.time() - t0 > budget or (a.max_steps and step >= a.max_steps):
                done = True
                break
        # save before the (slower) evaluation, so a killed session still leaves weights
        meta = {"labels": LABELS, "size": list(size), "mean": mean_l, "std": std_l, "arch": a.arch}
        torch.save({"state_dict": ema.state_dict(), **meta}, os.path.join(a.out, f"{name}_last.pt"))
        rep = {"epoch": ep, "step": step, "minutes": (time.time() - t0) / 60, "train_loss": tot / max(1, n)}
        for tag_, m in (("raw", net), ("ema", ema)):
            pm, yy = predict(m, hl, mean, std, dev, amp)
            rep[tag_] = metrics(pm, yy)
        pm, yy = predict(ema, ll, mean, std, dev, amp)
        rep["ema_lowres"] = metrics(pm, yy)
        hist.append(rep)
        print(json.dumps({k: (v["at_women_clear"] | {"auc": v["auc"]} if isinstance(v, dict) else v)
                          for k, v in rep.items()}), flush=True)
        for tag_, m in (("raw", net), ("ema", ema)):
            score = rep[tag_]["at_women_clear"]["0.005"]["men_left_clear"]
            if best is None or score > best[0]:
                best = (score, ep, tag_)
                torch.save({"state_dict": m.state_dict(), **meta}, os.path.join(a.out, f"{name}.pt"))
        json.dump({"best": {"men_left_clear_at_women_0.5pct": best[0], "epoch": best[1], "weights": best[2]},
                   "arch": a.arch, "tag": a.tag, "args": vars(a), "history": hist},
                  open(os.path.join(a.out, f"{name}_eval.json"), "w"), indent=1)
        ep += 1
    print(f"best: men left clear {best[0]:.4f} at 0.5% women (epoch {best[1]}, {best[2]})", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
PY = f"{TMP}/venv/bin/python"
if not os.path.exists(PY):
    run("pip install -q uv")
    run(f"uv venv -q -p 3.10 {TMP}/venv")
    run(f"uv pip install -q -p {PY} torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121")
    run(f"uv pip install -q -p {PY} timm==1.0.11 numpy==1.26.4 pillow==11.0.0 huggingface_hub==0.26.5 safetensors==0.4.5")
run(f"{PY} -c 'import torch, timm; print(torch.__version__, timm.__version__, torch.cuda.device_count(), \"GPUs\")'")
run("nproc; free -g; df -h /kaggle/tmp /kaggle/working")

In [ ]:
# box CSV (~2.3 GB): stream it and keep only the Woman/Man/Girl/Boy rows
CSV = f"{TMP}/train-wmgb.csv"
if not os.path.exists(CSV):
    run(f"curl -sfL {CSV_URL} | awk -F, 'NR==1 || $3==\"/m/03bt1vf\" || $3==\"/m/04yx4\" || $3==\"/m/05r655\" "
        f"|| $3==\"/m/01bl7v\"' > {CSV}.part && mv {CSV}.part {CSV}")
n = sum(1 for _ in open(CSV)) - 1
print(n, "rows")
assert n == CSV_ROWS, f"expected {CSV_ROWS} rows, got {n}"

In [ ]:
DATA = f"{TMP}/gcrops"
if not os.path.exists(f"{DATA}/index.csv"):
    run(f"{PY} models/vision/prep_train_crops.py --csv {CSV} --out {DATA} --per-side {PER_SIDE} "
        f"--max-minutes {PREP_MINUTES} --threads 12 2>{TMP}/prep_err.log")
run(f"tail -n 3 {TMP}/prep_err.log; grep -c skip {TMP}/prep_err.log || true; du -sh {DATA}")
run(f"cut -d, -f2,3 {DATA}/index.csv | sort | uniq -c")

In [ ]:
ngpu = int(subprocess.check_output([PY, "-c", "import torch; print(torch.cuda.device_count())"]).decode())
ncpu = os.cpu_count()
procs = []
if ngpu >= 2:   # one model per GPU, side by side
    for k, (arch, tag) in enumerate(ARCHS):
        log = open(f"{OUT}/gender_v2_{arch}.log", "w")
        cmd = [PY, "models/vision/train_gender_v2.py", "--data", DATA, "--out", OUT, "--arch", arch, "--tag", tag,
               "--minutes", str(TRAIN_MINUTES), "--workers", str(max(2, ncpu // 2 + 1))]
        procs.append((arch, subprocess.Popen(cmd, cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, "CUDA_VISIBLE_DEVICES": str(k)})))
    while any(p.poll() is None for _, p in procs):
        time.sleep(300)
        for arch, _ in procs:
            lines = open(f"{OUT}/gender_v2_{arch}.log").read().strip().splitlines()
            print(arch, "|", lines[-1][:300] if lines else "(no output yet)", flush=True)
    for arch, p in procs:
        print(arch, "exit", p.returncode)
else:           # one GPU: one after the other, half the budget each
    for arch, tag in ARCHS:
        run(f"{PY} models/vision/train_gender_v2.py --data {DATA} --out {OUT} --arch {arch} --tag {tag} "
            f"--minutes {TRAIN_MINUTES / 2} --workers {ncpu} 2>&1 | tee {OUT}/gender_v2_{arch}.log | grep -v '^ep '")

In [ ]:
for f in glob.glob(f"{OUT}/*_last.pt"):
    os.remove(f)
run(f"ls -la {OUT}; grep -h '^best' {OUT}/*.log || true")